# 00 · 실습 환경과 첫 응답

## 학습 목표
- Foundry, Agent Framework, Search, APIM의 역할을 구분합니다.
- VS Code에서 `.venv` 커널을 선택하고 개인 키로 실제 모델을 호출합니다.

## 개념
**Foundry**는 배포된 모델을 제공합니다. **Agent Framework**는 로컬 Python에서 대화·도구 실행 반복을 관리합니다. **AI Search**는 문서를 검색합니다. **APIM**은 강사의 공유 리소스를 개인 키별로 제한합니다. 학생은 Azure 계정이 필요 없습니다.

이 과정은 로컬 실행형 단일 에이전트입니다. Foundry Agent Service에 호스팅된 에이전트를 만드는 과정은 아닙니다. 별도 Foundry 프로젝트는 필요하지 않습니다.

### 준비
1. 저장소 루트에서 `uv sync --frozen` 실행.
2. 강사가 준 개인 `.env`를 저장소 루트에 저장. 다른 사람의 파일을 쓰지 마세요.
3. VS Code 우측 상단 **Select Kernel → Python Environments → .venv**.
4. 이 노트북부터 위에서 아래로 실행합니다. API 호출은 과금됩니다.

In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


### 1. 모델을 직접 호출하기
`gpt-5.6-terra`는 모델 별명이 아니라 강사가 실제 배포한 **배포명**입니다. API 키는 모델 키가 아닌 **본인 전용 APIM 키**입니다. `store=False`로 서버 측 응답 저장을 끕니다.

In [ ]:
async with create_openai_client(settings) as client:
    response = await client.responses.create(
        model=settings.model, input="RAG와 도구 호출의 차이를 한국어 두 문장으로 설명해줘.",
        store=False, max_output_tokens=250,
    )
    print(response.output_text)
    print("실제 사용량:", response.usage.model_dump() if response.usage else "서비스 미제공")
assert response.output_text


## 관찰
아직 문서를 검색하거나 날씨 API를 호출하지 않았습니다. 답변이 그럴듯해도 사내 규정이나 실시간 사실을 확인한 것이 아닙니다. 모델의 텍스트 답변과 실제 도구 실행은 다릅니다.

## 연습
질문을 “한빛테크 도쿄 숙박비 규정은?”으로 바꿨을 때 답을 신뢰할 수 있을까요? 실제 실행 전에 예측해 보세요.

<details><summary>예시 해설</summary>문서를 전달하지 않았으므로 정답 근거가 없습니다. 03~04에서 문서를 검색한 뒤 같은 질문과 비교합니다. 모르는 규정에 답했다면 그것은 검증된 답이 아닙니다.</details>

### 오류가 나면
401: 개인 키 확인 · 403: 배정 인덱스/키 조합 확인 · 429: 잠시 기다린 뒤 재실행 · 커널 import 오류: `uv sync --frozen`과 커널 선택 확인. 키나 예외 전체를 단체 채팅에 붙이지 마세요.